In [60]:
import pandas as pd

In [61]:
df = pd.read_csv("../data/processed/taxonomy_map.csv")
df

,image_path,class_id,kingdom,phylum,class,order,family,genus,scientific_name,common_name
0,ml_pipeline/data/raw/train_mini/02912_Animalia...,2912,Animalia,Chordata,Actinopterygii,Siluriformes,Ictaluridae,Ameiurus,Ameiurus nebulosus,Brown Bullhead Catfish
1,ml_pipeline/data/raw/train_mini/05804_Plantae_...,5804,Plantae,Tracheophyta,Liliopsida,Alismatales,Araceae,Peltandra,Peltandra virginica,Green Arrow Arum
2,ml_pipeline/data/raw/train_mini/00980_Animalia...,980,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Arctia,Arctia virginalis,Ranchman's Tiger Moth
3,ml_pipeline/data/raw/train_mini/09244_Plantae_...,9244,Plantae,Tracheophyta,Magnoliopsida,Ranunculales,Ranunculaceae,Delphinium,Delphinium parishii,desert larkspur
4,ml_pipeline/data/raw/train_mini/07200_Plantae_...,7200,Plantae,Tracheophyta,Magnoliopsida,Boraginales,Boraginaceae,Phacelia,Phacelia tanacetifolia,Lacy phacelia
...,...,...,...,...,...,...,...,...,...,...
499995,ml_pipeline/data/raw/train_mini/06091_Plantae_...,6091,Plantae,Tracheophyta,Liliopsida,Asparagales,Orchidaceae,Pterostylis,Pterostylis pedunculata,maroonhood
499996,ml_pipeline/data/raw/train_mini/05076_Animalia...,5076,Animalia,Chordata,Reptilia,Squamata,Scincidae,Plestiodon,Plestiodon fasciatus,Common Five-lined Skink
499997,ml_pipeline/data/raw/train_mini/01131_Animalia...,1131,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Trigonodes,Trigonodes hyppasia,Triangles
499998,ml_pipeline/data/raw/train_mini/02052_Animalia...,2052,Animalia,Arthropoda,Insecta,Lepidoptera,Pieridae,Appias,Appias libythea,Striped Albatross


In [62]:
df['class_id'].value_counts()

class_id
2912    50
6282    50
5848    50
4530    50
3612    50
        ..
5755    50
9664    50
1395    50
4666    50
8880    50
Name: count, Length: 10000, dtype: int64

In [63]:
from PIL import Image

In [64]:
n = 4096
df.iloc[n]

image_path         ml_pipeline/data/raw/train_mini/02575_Animalia...
class_id                                                        2575
kingdom                                                     Animalia
phylum                                                    Arthropoda
class                                                        Insecta
order                                                        Odonata
family                                                  Libellulidae
genus                                                      Sympetrum
scientific_name                                    Sympetrum vicinum
common_name                                        Autumn Meadowhawk
Name: 4096, dtype: object

In [65]:
Image.open(r"../../"+df.iloc[n].image_path)

FileNotFoundError: [Errno 2] No such file or directory: '../../ml_pipeline/data/raw/train_mini/02575_Animalia_Arthropoda_Insecta_Odonata_Libellulidae_Sympetrum_vicinum/b3f1b55e-7b67-438e-8add-5947bb96492b.jpg'

In [66]:
from pathlib import Path

# Real archive size on disk vs. official ~47 GB
print(Path("../../ml_pipeline/data/raw/train_mini.tar.gz").stat().st_size / 1e9, "GB")

# How many species directories actually got extracted
dirs = list(Path("../../ml_pipeline/data/raw/train_mini").iterdir())
print(len(dirs), "/ 10000 species directories on disk")

# Is 02575 in there at all?
print(list(Path("../../ml_pipeline/data/raw/train_mini").glob("02575_*")))

# Whole-CSV existence check
df["exists"] = df["image_path"].map(lambda p: Path("../../" + p).exists())
print(df["exists"].mean(), "fraction of CSV rows backed by a real file")

44.636137542 GB
6218 / 10000 species directories on disk
[]
0.621742 fraction of CSV rows backed by a real file


In [67]:
from pathlib import Path
dirs = sorted(p.name for p in Path("../../ml_pipeline/data/raw/train_mini").iterdir())
print("first:", dirs[0])
print("last :", dirs[-1])
print("count:", len(dirs))

# Are the present directories a contiguous prefix of class ids,
# or scattered randomly?
ids = sorted(int(d.split("_", 1)[0]) for d in dirs)
gaps = [(a, b) for a, b in zip(ids, ids[1:]) if b - a > 1]
print("number of gaps:", len(gaps), "first 3 gaps:", gaps[:3])

first: 00000_Animalia_Annelida_Clitellata_Haplotaxida_Lumbricidae_Lumbricus_terrestris
last : 09994_Plantae_Tracheophyta_Polypodiopsida_Polypodiales_Woodsiaceae_Woodsia_obtusa
count: 6218
number of gaps: 2372 first 3 gaps: [(2, 4), (5, 7), (7, 9)]


In [76]:
import hashlib

def verify_checksum(file_path, expected_hash):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        # Read in chunks so it doesn't crash your RAM
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    
    actual_hash = hash_md5.hexdigest()
    if actual_hash == expected_hash:
        print("✅ Match! The file is perfect.")
    else:
        print(f"❌ Mismatch! \nActual: {actual_hash}\nExpected: {expected_hash}")

verify_checksum(r"..\..\ml_pipeline\data\raw\train_mini.tar.gz", "db6ed8330e634445efc8fec83ae81442")


❌ Mismatch! 
Actual: 3da3754eff27d3cd6c41daaeb3f6fc89
Expected: db6ed8330e634445efc8fec83ae81442


In [77]:
import hashlib

def verify_checksum(file_path, expected_hash):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        # Read in chunks so it doesn't crash your RAM
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    
    actual_hash = hash_md5.hexdigest()
    if actual_hash == expected_hash:
        print("✅ Match! The file is perfect.")
    else:
        print(f"❌ Mismatch! \nActual: {actual_hash}\nExpected: {expected_hash}")

verify_checksum(r"..\..\ml_pipeline\data\raw\train_mini.json.tar.gz", "395a35be3651d86dc3b0d365b8ea5f92")



✅ Match! The file is perfect.
